# Real Image Editing — One-Shot Colab Notebook

End-to-end run on **one image + one instruction**, producing two paths and three visualizations.

**Method path** (PEZ + P2P):
  1. PEZ-1 inversion → discrete source prompt + per-timestep null-text
  2. PEZ-2 instruction-conditioned target prompt sweep over Knob 1 (`λ_instruction`, `γ_anchor`)
  3. P2P + LocalBlend edit sweep over Knob 2 (`cross_replace_steps`)

**Baseline path** (BLIP + manual prompt):
  1. BLIP captions the source
  2. User-supplied target prompt (set after seeing the caption)
  3. Standard P2P + LocalBlend edit over Knob 2

**Visualizations:**
  1. Prompt evolution PEZ-1 → PEZ-2 across Knob 1
  2. 2D matrix: rows = Knob 1 (PEZ-2 prompts), cols = Knob 2 (cross_replace_steps)
  3. 1D row: BLIP-source + manual-target P2P, cols = same Knob 2 axis as #2

**Outputs saved to `outputs/colab_oneshot/`:** source.png, every grid cell as an individual PNG, the composed Vis #2 / Vis #3 figures, the Vis #1 prompt-diff HTML, and a `run_metadata.json` with all prompts, knob settings, BLIP caption, and model IDs.

**Recommended run order:** the BLIP-caption cell runs in seconds and prints the caption *before* PEZ-1 (~70 min) starts, so you can write a good `MANUAL_TARGET_PROMPT` while looking at the caption.

**Expected runtime on A100:** ~90–150 min cold (PEZ-1 dominates). PEZ-1 and PEZ-2 results are cached, so re-runs with the same image/instruction are fast.

**GPU:** A100 (40 GB) recommended. Cold weights total ~10 GB (SD2.1 + OpenCLIP-H/14 + BLIP); T4 (15 GB) may OOM during PEZ-1 backward.

**Requires:** `HF_TOKEN` Colab secret (key icon in left sidebar) for SD2.1 weights.

In [ ]:
# 1. Clone repo and install deps not already on Colab
import os
if not os.path.isdir('real-image-editing'):
    !git clone https://github.com/beratcelik1/real-image-editing.git
%cd real-image-editing
!git submodule update --init --recursive 2>/dev/null
!pip install -q diffusers transformers accelerate safetensors Pillow scikit-image matplotlib pandas

import torch
assert torch.cuda.is_available(), 'GPU runtime required (Runtime → Change runtime type → GPU)'
print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 2. HuggingFace login
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print('Logged in via Colab secret')
except Exception:
    if os.environ.get('HF_TOKEN'):
        login(token=os.environ['HF_TOKEN'])
        print('Logged in via env var')
    else:
        print('WARNING: No HF_TOKEN found. SD2.1 download may fail.')

In [ ]:
# 3. Inputs — change these for your run.
#    The MANUAL_TARGET_PROMPT for the BLIP baseline lives in its own cell
#    further down so you can set it AFTER seeing the BLIP caption.

# 3a. Source image (URL or upload to ./data/)
IMAGE_URL = 'https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=512'
IMAGE_PATH = 'data/sample.jpg'
os.makedirs('data', exist_ok=True)
if not os.path.isfile(IMAGE_PATH):
    !wget -q -O {IMAGE_PATH} "{IMAGE_URL}"

# 3b. Method-path edit instruction (used by PEZ-2)
INSTRUCTION = 'change the cat into a dog'

# 3c. Knob 1 grid — PEZ-2 hyperparameters. Each row of vis #2.
KNOB_1_GRID = {
    'conservative': {'lambda_instruction': 0.5, 'gamma_anchor': 1.0},
    'moderate':     {'lambda_instruction': 1.0, 'gamma_anchor': 0.1},
    'aggressive':   {'lambda_instruction': 3.0, 'gamma_anchor': 0.01},
}

# 3d. Knob 2 grid — cross_replace_steps. Common axis between vis #2 and #3.
KNOB_2_GRID = [0.8, 0.5, 0.3]

# 3e. Output directory (everything saved here for offline review)
from pathlib import Path
OUTPUT_DIR = Path('outputs/colab_oneshot')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from PIL import Image
Image.open(IMAGE_PATH).resize((256, 256))

In [ ]:
# 4. Load models: SD2.1 + CLIP (for PEZ-1 image features) + BLIP (for caption baseline)
#
# CLIP-variant choice: SD2.1-base's text encoder is OpenCLIP-H/14 (hidden_size=1024).
# The CLIP loaded here MUST be 1024-dim or `clip_similarity_loss` errors on the
# cosine-similarity dot product (SD-pooled vs. clip_image_features). We use the
# HF-compatible OpenCLIP-H/14 checkpoint from LAION.
from src.utils import (
    load_sd_components, load_image, encode_image, decode_latent,
    get_text_embeddings, get_uncond_embeddings, get_device,
)
from src.config import load_pez_1, load_pez_2, load_local_blend, load_edit

device = get_device()

edit_config = load_edit()
print(f'Loading {edit_config.sd_model}...')
unet, vae, text_encoder, tokenizer, scheduler = load_sd_components(
    model_id=edit_config.sd_model,
)
sd_components = {
    'unet': unet, 'vae': vae, 'text_encoder': text_encoder,
    'tokenizer': tokenizer, 'scheduler': scheduler,
}

# OpenCLIP-H/14 (HF format) — matches SD2.1-base's 1024-dim text encoder.
from transformers import CLIPModel, CLIPProcessor
CLIP_HF_ID = 'laion/CLIP-ViT-H-14-laion2B-s32B-b79K'
clip_model = CLIPModel.from_pretrained(CLIP_HF_ID).to(device).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_HF_ID)
text_projection = clip_model.text_projection  # nn.Linear(1024, 1024)

# BLIP for the baseline caption
from transformers import BlipForConditionalGeneration, BlipProcessor
BLIP_HF_ID = 'Salesforce/blip-image-captioning-large'
blip_processor = BlipProcessor.from_pretrained(BLIP_HF_ID)
blip_model = BlipForConditionalGeneration.from_pretrained(BLIP_HF_ID).to(device).eval()

source_image = load_image(IMAGE_PATH, size=512)
with torch.no_grad():
    inputs = clip_processor(images=source_image, return_tensors='pt').to(device)
    clip_image_features = clip_model.get_image_features(**inputs)

# Free the rest of the CLIP vision/text weights from VRAM. text_projection was
# captured above as a separate reference and stays alive on device.
del clip_model
torch.cuda.empty_cache()
print('Models loaded.')

In [ ]:
# 5. BLIP caption — runs in seconds. Look at the printed caption, then
#    set MANUAL_TARGET_PROMPT in the next cell. Doing this BEFORE PEZ-1
#    means you decide the baseline target while PEZ-1 is still queued.
with torch.no_grad():
    inputs = blip_processor(images=source_image, return_tensors='pt').to(device)
    out = blip_model.generate(**inputs, max_length=30)
BLIP_SOURCE_PROMPT = blip_processor.decode(out[0], skip_special_tokens=True)
print(f'BLIP caption: {BLIP_SOURCE_PROMPT!r}')
print('\n→ Edit MANUAL_TARGET_PROMPT in the next cell based on this caption.')

# Free BLIP weights — we already have the caption string.
del blip_model, blip_processor
torch.cuda.empty_cache()

In [ ]:
# 6. Manual target prompt for the BLIP baseline path. Edit this AFTER
#    seeing the caption above. The baseline runs `BLIP_SOURCE_PROMPT →
#    MANUAL_TARGET_PROMPT` through standard P2P (no PEZ).
MANUAL_TARGET_PROMPT = 'a photo of a dog looking at the camera'
print(f'Source (from BLIP): {BLIP_SOURCE_PROMPT!r}')
print(f'Target (manual):    {MANUAL_TARGET_PROMPT!r}')

---
## Phase A — Method path (PEZ-1 → PEZ-2 → P2P)

In [ ]:
# 7. PEZ-1: discrete source-prompt inversion (cached on disk)
from src.pez.source_inversion import pez_invert_source

print('Running PEZ-1 (~70 min on first run, instant on cache hit)...')
pez_1_config = load_pez_1()
source_token_ids, null_text_per_timestep = pez_invert_source(
    image=source_image,
    config=pez_1_config,
    sd_components=sd_components,
    clip_image_features=clip_image_features,
    text_projection=text_projection,
)
pez_1_prompt = tokenizer.decode(source_token_ids, skip_special_tokens=True)
print(f'PEZ-1 prompt: {pez_1_prompt!r}')

In [ ]:
# 8. PEZ-2 sweep over Knob 1 (cached per-setting)
from dataclasses import replace
from src.pez.instruction_conditioned import pez_invert_with_instruction

pez_2_config = load_pez_2()
pez_2_results = {}  # name -> (target_token_ids, decoded_prompt)

for name, overrides in KNOB_1_GRID.items():
    print(f'\n[PEZ-2 {name}] {overrides}')
    cfg = replace(pez_2_config, **overrides)
    target_token_ids = pez_invert_with_instruction(
        image=source_image,
        instruction=INSTRUCTION,
        pez_1_token_ids=source_token_ids,
        null_text_per_timestep=null_text_per_timestep,
        config=cfg,
        sd_components=sd_components,
    )
    target_prompt = tokenizer.decode(target_token_ids, skip_special_tokens=True)
    pez_2_results[name] = (target_token_ids, target_prompt)
    print(f'  → {target_prompt!r}')

In [ ]:
# 9. Visualization 1 — prompt evolution PEZ-1 → PEZ-2 across Knob 1
import difflib
import pandas as pd
from IPython.display import display, HTML

rows = [{
    'Setting': 'PEZ-1 (source)', 'λ_instr': '—', 'γ_anchor': '—',
    'Decoded prompt': pez_1_prompt,
}]
for name, overrides in KNOB_1_GRID.items():
    _, prompt = pez_2_results[name]
    rows.append({
        'Setting': f'PEZ-2 ({name})',
        'λ_instr': overrides['lambda_instruction'],
        'γ_anchor': overrides['gamma_anchor'],
        'Decoded prompt': prompt,
    })
vis1_df = pd.DataFrame(rows)
display(vis1_df)

# Inline token-level diff against PEZ-1 baseline
def render_diff(base_tokens, new_tokens):
    sm = difflib.SequenceMatcher(None, base_tokens, new_tokens)
    parts = []
    for op, i1, i2, j1, j2 in sm.get_opcodes():
        if op == 'equal':
            parts.append(f"<span style='color:#888'>{' '.join(base_tokens[i1:i2])}</span>")
        elif op == 'replace':
            parts.append(
                f"<span style='color:#c33;text-decoration:line-through'>{' '.join(base_tokens[i1:i2])}</span> "
                f"<span style='color:#0a0;font-weight:bold'>{' '.join(new_tokens[j1:j2])}</span>"
            )
        elif op == 'insert':
            parts.append(f"<span style='color:#0a0;font-weight:bold'>+{' '.join(new_tokens[j1:j2])}</span>")
        elif op == 'delete':
            parts.append(f"<span style='color:#c33;text-decoration:line-through'>{' '.join(base_tokens[i1:i2])}</span>")
    return ' '.join(parts)

base = pez_1_prompt.split()
html_parts = [
    f"<h3>Vis #1 — PEZ-1 → PEZ-2 prompt evolution</h3>",
    f"<p><b>Instruction:</b> {INSTRUCTION!r}</p>",
    f"<p><b>PEZ-1 (source):</b> {pez_1_prompt}</p>",
    "<h4>Token-level diff vs PEZ-1</h4>",
]
for name in KNOB_1_GRID:
    _, p2 = pez_2_results[name]
    html_parts.append(f"<p><b>{name}:</b> {render_diff(base, p2.split())}</p>")
vis1_html = '\n'.join(html_parts)
display(HTML(vis1_html))

# Persist Vis #1 — full self-contained HTML file with the table embedded.
vis1_full_html = (
    '<!doctype html><meta charset="utf-8">'
    '<style>body{font-family:sans-serif;max-width:900px;margin:2em auto;padding:0 1em}'
    'table{border-collapse:collapse;margin:1em 0}td,th{padding:.4em .8em;border:1px solid #ddd}</style>'
    f'{vis1_html}<h4>Table</h4>{vis1_df.to_html(index=False)}'
)
(OUTPUT_DIR / 'vis1_prompt_evolution.html').write_text(vis1_full_html, encoding='utf-8')
print(f"Saved Vis #1 → {OUTPUT_DIR / 'vis1_prompt_evolution.html'}")

In [ ]:
# 10. Helper: run the editing portion only — given source/target token IDs,
#     invert the source under its prompt and run P2P + LocalBlend with the
#     requested cross_replace_steps. Used by both the method path (Knob 1×2
#     grid below) and the BLIP baseline path further down.
from src.inversion import ddim_inversion
from src.splice.align import align_pez_prompts
from src.pipeline import _run_editing_loop, _apply_overrides_nested
from attention_control.cross_attention import (
    CrossAttentionController, register_attention_control, unregister_attention_control,
)
from attention_control.local_blend import LocalBlend


def run_edit(source_token_ids, target_token_ids, cross_replace_steps):
    """DDIM-invert source, then run a P2P+LocalBlend edit at the requested
    cross_replace_steps. Returns the edited PIL image."""
    edit_cfg = load_edit()
    edit_cfg = _apply_overrides_nested(
        edit_cfg,
        {'cross_attention.cross_replace_steps': cross_replace_steps},
    )
    lb_cfg = load_local_blend()

    source_str = tokenizer.decode(source_token_ids, skip_special_tokens=True)
    target_str = tokenizer.decode(target_token_ids, skip_special_tokens=True)
    src_emb = get_text_embeddings(source_str, tokenizer, text_encoder, device)
    tgt_emb = get_text_embeddings(target_str, tokenizer, text_encoder, device)
    uncond  = get_uncond_embeddings(tokenizer, text_encoder, device)

    image_latent = encode_image(source_image, vae, device)
    z_T, _ = ddim_inversion(
        image_latent, src_emb, uncond, unet, scheduler,
        num_steps=edit_cfg.ddim.num_steps, cfg_scale=edit_cfg.ddim.cfg_scale,
    )

    full_src = tokenizer.encode(source_str)
    full_tgt = tokenizer.encode(target_str)
    mapping, unmapped = align_pez_prompts(full_src, full_tgt, method=edit_cfg.alignment_method)

    local_blend = None
    if lb_cfg.enabled and unmapped:
        local_blend = LocalBlend(
            target_token_indices=unmapped,
            threshold=lb_cfg.threshold,
            base_resolution=lb_cfg.base_resolution,
            dilate_iters=lb_cfg.dilate_iters,
        )

    layer_indices = (
        set(edit_cfg.cross_attention.layer_indices)
        if edit_cfg.cross_attention.layer_indices else None
    )
    cross_ctrl = CrossAttentionController(
        num_steps=edit_cfg.ddim.num_steps,
        cross_replace_steps=edit_cfg.cross_attention.cross_replace_steps,
        token_mapping=mapping,
        layer_indices=layer_indices,
        local_blend=local_blend,
    )
    register_attention_control(unet, cross_ctrl)
    edited_latent = _run_editing_loop(
        z_T=z_T, source_emb=src_emb, target_emb=tgt_emb, uncond_emb=uncond,
        unet=unet, scheduler=scheduler, cross_ctrl=cross_ctrl,
        cfg_scale=edit_cfg.ddim.cfg_scale, num_steps=edit_cfg.ddim.num_steps,
    )
    unregister_attention_control(unet)
    return decode_latent(edited_latent[1:2], vae)

In [ ]:
# 11. Run the 2D grid: Knob 1 (PEZ-2 settings) × Knob 2 (cross_replace_steps)
grid_results = {}
for k1_name in KNOB_1_GRID:
    target_ids, _ = pez_2_results[k1_name]
    for k2_val in KNOB_2_GRID:
        print(f'Edit [{k1_name}, cross_replace={k2_val}]')
        grid_results[(k1_name, k2_val)] = run_edit(
            source_token_ids, target_ids, k2_val,
        )

In [ ]:
# 12. Visualization 2 — 2D matrix. Saves the composed figure as a PNG
#     in addition to the per-cell PNGs written by the final save cell.
import matplotlib.pyplot as plt

rows, cols = len(KNOB_1_GRID), len(KNOB_2_GRID)
fig, axes = plt.subplots(rows, cols, figsize=(3.5 * cols, 3.5 * rows), squeeze=False)
for i, k1_name in enumerate(KNOB_1_GRID):
    overrides = KNOB_1_GRID[k1_name]
    for j, k2_val in enumerate(KNOB_2_GRID):
        ax = axes[i, j]
        ax.imshow(grid_results[(k1_name, k2_val)])
        if i == 0:
            ax.set_title(f'cross_replace={k2_val}')
        if j == 0:
            ax.set_ylabel(
                f"{k1_name}\nλ={overrides['lambda_instruction']}\nγ={overrides['gamma_anchor']}",
                fontsize=10,
            )
        ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f'Method (PEZ-1 + PEZ-2 + P2P) — instruction: {INSTRUCTION!r}', fontsize=13)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'vis2_method_grid.png', dpi=120, bbox_inches='tight')
print(f"Saved Vis #2 → {OUTPUT_DIR / 'vis2_method_grid.png'}")
plt.show()

---
## Phase B — BLIP baseline path (caption + manual target → standard P2P)

In [ ]:
# 13. Tokenize the BLIP source caption + manual target. (BLIP caption was
#     generated way back in cell 5; MANUAL_TARGET_PROMPT was set in cell 6.)
blip_src_ids = tokenizer.encode(BLIP_SOURCE_PROMPT, add_special_tokens=False)
blip_tgt_ids = tokenizer.encode(MANUAL_TARGET_PROMPT, add_special_tokens=False)
print(f'Source ({len(blip_src_ids)} tokens): {BLIP_SOURCE_PROMPT!r}')
print(f'Target ({len(blip_tgt_ids)} tokens): {MANUAL_TARGET_PROMPT!r}')

In [ ]:
# 14. 1D sweep — BLIP source + manual target across the same Knob 2 axis as vis #2
blip_results = {}
for k2_val in KNOB_2_GRID:
    print(f'BLIP edit [cross_replace={k2_val}]')
    blip_results[k2_val] = run_edit(blip_src_ids, blip_tgt_ids, k2_val)

In [ ]:
# 15. Visualization 3 — 1D BLIP edit row (shares Knob 2 axis with vis #2).
#     Saves the composed figure as a PNG.
fig, axes = plt.subplots(1, len(KNOB_2_GRID), figsize=(3.5 * len(KNOB_2_GRID), 3.5), squeeze=False)
for j, k2_val in enumerate(KNOB_2_GRID):
    ax = axes[0, j]
    ax.imshow(blip_results[k2_val])
    ax.set_title(f'cross_replace={k2_val}')
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(
    f'Baseline (BLIP source + manual target)\n'
    f'src: {BLIP_SOURCE_PROMPT!r}\n'
    f'tgt: {MANUAL_TARGET_PROMPT!r}',
    fontsize=11,
)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'vis3_baseline_row.png', dpi=120, bbox_inches='tight')
print(f"Saved Vis #3 → {OUTPUT_DIR / 'vis3_baseline_row.png'}")
plt.show()

In [ ]:
# 16. Save per-cell PNGs + run_metadata.json. Combined with the composed
#     figures (cells 12, 15) and the prompt-evolution HTML (cell 9), the
#     output directory is a self-contained record of the run.
import json
import datetime

source_image.save(OUTPUT_DIR / 'source.png')
for (k1, k2), img in grid_results.items():
    img.save(OUTPUT_DIR / f'method_{k1}_xrep{k2}.png')
for k2, img in blip_results.items():
    img.save(OUTPUT_DIR / f'baseline_xrep{k2}.png')

metadata = {
    'timestamp_utc': datetime.datetime.utcnow().isoformat() + 'Z',
    'image_path': IMAGE_PATH,
    'image_url': IMAGE_URL,
    'instruction': INSTRUCTION,
    'manual_target_prompt': MANUAL_TARGET_PROMPT,
    'blip_caption': BLIP_SOURCE_PROMPT,
    'pez_1': {
        'token_ids': list(map(int, source_token_ids)),
        'decoded_prompt': pez_1_prompt,
    },
    'pez_2_results': {
        name: {
            'lambda_instruction': KNOB_1_GRID[name]['lambda_instruction'],
            'gamma_anchor': KNOB_1_GRID[name]['gamma_anchor'],
            'token_ids': list(map(int, target_ids)),
            'decoded_prompt': prompt,
        }
        for name, (target_ids, prompt) in pez_2_results.items()
    },
    'knob_1_grid': KNOB_1_GRID,
    'knob_2_grid': list(KNOB_2_GRID),
    'model_ids': {
        'sd': edit_config.sd_model,
        'clip': CLIP_HF_ID,
        'blip': BLIP_HF_ID,
    },
    'configs': {
        'pez_1': pez_1_config.__dict__,
        'pez_2_base': {k: v for k, v in pez_2_config.__dict__.items()
                       if k not in {'lambda_instruction', 'gamma_anchor'}},
        'edit': {
            'sd_model': edit_config.sd_model,
            'ddim': edit_config.ddim.__dict__,
            'cross_attention_base': edit_config.cross_attention.__dict__,
            'alignment_method': edit_config.alignment_method,
        },
        'local_blend': load_local_blend().__dict__,
    },
    'output_files': {
        'source': 'source.png',
        'method_grid_cells': [f'method_{k1}_xrep{k2}.png' for k1 in KNOB_1_GRID for k2 in KNOB_2_GRID],
        'baseline_cells':   [f'baseline_xrep{k2}.png' for k2 in KNOB_2_GRID],
        'vis1_html':        'vis1_prompt_evolution.html',
        'vis2_composed':    'vis2_method_grid.png',
        'vis3_composed':    'vis3_baseline_row.png',
    },
}
(OUTPUT_DIR / 'run_metadata.json').write_text(json.dumps(metadata, indent=2, default=str), encoding='utf-8')

n_pngs = 1 + len(grid_results) + len(blip_results) + 2  # +2 for vis2/vis3 composed
print(f'Saved {n_pngs} PNGs + 1 HTML + 1 JSON to {OUTPUT_DIR}/')
print('Files:')
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {p.name}')